# LogSeer Prediction

Predicts whether a planned operation will succeed or fail based on current server logs.

**Setup**: set `DATA_DIR` to a directory of log files (single set) or a directory of subdirectories (one per operation run, prediction runs on each). Set the model/tokenizer paths to your trained files — if you trained locally they should already be in the parent directory; if you trained on Colab, copy them from Google Drive (see cell below).

In [57]:
import os, sys

# On Google Colab: clone the repo and set up the path
# On local: assumes notebook is run from the notebooks/ directory
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('logseer'):
        os.system('git clone https://github.com/masahiko-shibata/logseer.git')
    os.chdir('logseer/notebooks')
sys.path.insert(0, os.path.abspath('..'))

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
from logseer import Seer, print_results

In [64]:
# Configuration
BASE_DIR            = '../'                           # directory where trained models are stored
DATA_DIR            = '../logs/error'
NN_MODEL_FILE       = 'logseer.keras'
TOKENIZER_FILE      = 'tokenizer.pickle'
SK_MODEL_FILE       = 'xgb.pkl'
NN_MODEL_PATH       = BASE_DIR + NN_MODEL_FILE
TOKENIZER_PATH      = BASE_DIR + TOKENIZER_FILE
SK_MODEL_PATH       = BASE_DIR + SK_MODEL_FILE
NUMCHAR             = 3000
MAX_SEQUENCE_LENGTH = 26000
NN_THRESHOLD        = 0.72
XGB_THRESHOLD       = 0.52

In [59]:
# Google Colab only - Run this to copy data and trained models from Google Drive. Otherwise, copy the data folder to DATA_DIR.
from google.colab import drive
import shutil, zipfile
drive.mount('/content/drive')

DRIVE_BASE      = '/content/drive/MyDrive/Colab Notebooks/logseer/'
DRIVE_DATA_DIR  = DRIVE_BASE + '/data/'
DRIVE_MODEL_DIR = DRIVE_BASE + '/models/'
DATA_ZIP        = 'data_current.zip'

shutil.copy(DRIVE_MODEL_DIR + NN_MODEL_FILE,  NN_MODEL_PATH)
shutil.copy(DRIVE_MODEL_DIR + TOKENIZER_FILE, TOKENIZER_PATH)
shutil.copy(DRIVE_MODEL_DIR + SK_MODEL_FILE,  SK_MODEL_FILE)
shutil.copy(DRIVE_DATA_DIR + DATA_ZIP,  BASE_DIR + DATA_ZIP)
with zipfile.ZipFile(BASE_DIR + DATA_ZIP, 'r') as z:
    z.extractall(BASE_DIR)
print(os.listdir(BASE_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['requirements.txt', 'example_test_result.txt', 'predict.py', 'tune_threshold.py', 'train.py', 'data_current.zip', 'logseer.keras', 'LICENSE', 'notebooks', 'tokenizer.pickle', 'README.md', '.gitignore', 'logseer', 'config.yaml', '.git', 'logs']


Run the cell below to load models and predict. Each row is one log set. `OR` triggers on either model, `AND` requires both — use AND as the high-precision restart signal.

In [66]:
# Load models and predict
seer    = Seer.from_files(NN_MODEL_PATH, TOKENIZER_PATH, SK_MODEL_FILE,
                          numchar=NUMCHAR, max_sequence_length=MAX_SEQUENCE_LENGTH,
                          nn_threshold=NN_THRESHOLD, xgb_threshold=XGB_THRESHOLD)
results = seer.predict(DATA_DIR)
print_results(results)

*** Generating Data ***
Processing... [##################################################] 100% 
*** Cleaning Data ***
Processing... [##################################################] 100% 
  Set    NN_prob  XGB_prob     OR    AND
  ----------------------------------------
  1046    0.9129    0.9962    ERR    ERR
  1102    0.1788    0.0041     ok     ok
  1103    0.9090    0.9922    ERR    ERR
  1104    0.9920    0.9987    ERR    ERR
  1116    0.3613    0.9869    ERR     ok
  1117    0.6314    0.9777    ERR     ok
  1278    0.6217    0.9971    ERR     ok
  1283    0.8723    0.9964    ERR    ERR
  1284    0.1450    0.9934    ERR     ok
  1285    0.9092    0.9936    ERR    ERR
  1289    0.8294    0.9984    ERR    ERR
  1290    0.3619    0.9957    ERR     ok
  1344    0.7618    0.9927    ERR    ERR
  1355    0.9497    0.9960    ERR    ERR
  1403    0.4749    0.9938    ERR     ok
  1407    0.2552    0.9861    ERR     ok
  1504    0.9476    0.9939    ERR    ERR
  1508    0.9691    0.9966 